In [1]:
import pandas as pd
import os
import numpy as np
from datetime import datetime
from zoneinfo import ZoneInfo
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option('display.max_columns', None)

In [40]:
df_desgravamen= pd.read_excel('C:/DATOS/errores desgravamane Efectiva - Luis Villena - 2026 v2.xlsx', sheet_name=0,
                              dtype={'NRO LOTES ANTERIORES': str, 'CÓDIGO ERROR': str, 'PLACA': str, 
                                     'MOTOR': str, 'TIPO DOC. ID': str, 'DOC. ID.': str, 'NOMBRE': str,
                                     'APELLIDO_PATERNO':str,'APELLIDO MATERNO': str})
df_desgravamen.columns = (df_desgravamen.columns.str.strip().str.upper()
                            .str.replace(r'[^A-Za-z0-9]', '_', regex=True))



In [41]:
df_desgravamen.drop(['L_NEA_DE_TRAMA'], axis=1, inplace=True)

In [42]:
df_desgravamen= df_desgravamen.rename(columns={'C_DIGO_PRODUCTO':'CODIGO_PRODUCTO','C_D__CERTIFICADO_CANAL':'CODIGO_CERTIFICADO',
                                               'C_DIGO_ERROR':'CODIGO_ERROR', 'DESCRIPCI_N_ERROR':'DESCRIPCION_ERROR',
                                                'VALIDACI_N_APLICADA':'VALIDACION_APLICADA', 'DOC__ID_':'DOC_ID', 'TIPO_DOC__ID': 'TIPO_DOC_ID',
                                                'FEC__INICIO':'FECHA_INICIO', 'FEC__FIN':'FECHA_FIN', 'PRIMABRUTACAN':'PRIMA_BRUTA', 
                                                'PRIMANETACAN':'PRIMA_NETA', 'LINEA_TRAMA_TEXTO_COMPLETO_':'LINEA_TRAMA'})

In [43]:
df_desgravamen.head(3)

,NRO_LOTE,NRO_LOTES_ANTERIORES,FECHA_CARGA_LOTE,CODIGO_PRODUCTO,PRODUCTO,PLAN,NOMBRE_DE_PLAN,CODIGO_CERTIFICADO,NRO,IDEDET,TIPO_MOVIMIENTO,ORIGEN_ERROR,CODIGO_ERROR,DESCRIPCION_ERROR,NOMBRE,APELLIDO_PATERNO,APELLIDO_MATERNO,FECHA_NACIMIENTO,TIPO_DOC_ID,DOC_ID,FECHA_INICIO,FECHA_FIN,MONEDA,SUMA_ASEGURADA,TASA,TASA_CON_RECARGO,PRIMA_BRUTA,PRIMA_NETA,PLACA,MOTOR,NOMBRE_ARCHIVO,LINEA_TRAMA
0,1374146,726598|1304277|1313161|1315944|1332255|1340704...,2026-01-29 23:00:09,3071,Desgravamen EFECTIVA,60075,Plan Convencional,C00000000000001000248,1,857195393,Renovacion,Error Canal,8016,EL CAMPO TASA SIN RECARGO NO CORRESPONDE AL PL...,JUDITH JANET,GONZALES,GUEVARA,1991-09-02 00:00:00,2,47503836,2021-10-22,2021-11-22,SOL,10000.5,0.0,0.0,36.25,36.25,NaN,NaN,20441805960_0103001_20260105_506.TXT,980000000000000010002480000000000000100024801P...
1,1374146,726598|1304277|1313161|1315944|1332255|1340704...,2026-01-29 23:00:09,3071,Desgravamen EFECTIVA,60075,Plan Convencional,C00000000000001000319,1,857195394,Renovacion,Error Canal,8016,EL CAMPO TASA SIN RECARGO NO CORRESPONDE AL PL...,JUANA,CHILE,CUBA,1973-10-20 00:00:00,2,10401109,2021-10-22,2021-11-22,SOL,20604.6,0.0,0.0,66.39,66.39,NaN,NaN,20441805960_0103001_20260105_506.TXT,980000000000000010003190000000000000100031901P...
2,1374146,726598|1304277|1313161|1315944|1332255|1340704...,2026-01-29 23:00:09,3071,Desgravamen EFECTIVA,60075,Plan Convencional,C00000000000001000743,1,857195395,Renovacion,Error Rimac,1153,EL ALTA DEL NRO. CERTIFICADO EXTERNO: 00000000...,EVER,RIVERA,SOTO,1980-10-29 00:00:00,2,40873759,2021-10-30,2021-11-30,SOL,20091.0,0.0,0.0,83.23,83.23,NaN,NaN,20441805960_0103001_20260105_506.TXT,980000000000000010007430000000000000100074301P...


In [37]:
df_desgravamen['PRODUCTO'] = df_desgravamen["PRODUCTO"].str.upper()
df_desgravamen['TIPO_MOVIMIENTO'] = df_desgravamen["TIPO_MOVIMIENTO"].str.upper()
df_desgravamen['ORIGEN_ERROR'] = df_desgravamen["ORIGEN_ERROR"].str.upper()
df_desgravamen['CODIGO_CERTIFICADO'] = df_desgravamen["CODIGO_CERTIFICADO"].str[1:]

df_desgravamen['FECHA_CARGA_LOTE']= pd.to_datetime(df_desgravamen['FECHA_CARGA_LOTE'], errors='coerce').dt.date
df_desgravamen['FECHA_NACIMIENTO']= pd.to_datetime(df_desgravamen['FECHA_NACIMIENTO'], format='%Y-%m-%d', errors='coerce').dt.date
df_desgravamen['FECHA_INICIO']= pd.to_datetime(df_desgravamen['FECHA_INICIO'], errors='coerce').dt.date
df_desgravamen['FECHA_FIN']= pd.to_datetime(df_desgravamen['FECHA_FIN'], errors='coerce').dt.date

In [38]:
df_desgravamen["TIPO_SEGURO"] = df_desgravamen["LINEA_TRAMA"].str[:3]
df_desgravamen["CODIGO_MOVIMIENTO"] = df_desgravamen["LINEA_TRAMA"].str[48]
df_desgravamen["TIPO_REGISTRO"] = df_desgravamen["LINEA_TRAMA"].str[43:45]

df_desgravamen['FECHA_CARGA'] = datetime.now(ZoneInfo("America/Lima")).strftime("%Y-%m-%d")
df_desgravamen['FECHA_CARGA'] = pd.to_datetime(df_desgravamen['FECHA_CARGA'], errors='coerce').dt.date

In [39]:
df_desgravamen.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   NRO_LOTE              1500 non-null   int64  
 1   NRO_LOTES_ANTERIORES  1304 non-null   str    
 2   FECHA_CARGA_LOTE      1500 non-null   object 
 3   CODIGO_PRODUCTO       1500 non-null   int64  
 4   PRODUCTO              1500 non-null   str    
 5   PLAN                  1500 non-null   int64  
 6   NOMBRE_DE_PLAN        1489 non-null   str    
 7   CODIGO_CERTIFICADO    1500 non-null   str    
 8   NRO                   1500 non-null   int64  
 9   IDEDET                1500 non-null   int64  
 10  TIPO_MOVIMIENTO       1500 non-null   str    
 11  ORIGEN_ERROR          1500 non-null   str    
 12  CODIGO_ERROR          1500 non-null   str    
 13  DESCRIPCION_ERROR     1500 non-null   str    
 14  NOMBRE                1500 non-null   str    
 15  APELLIDO_PATERNO      1500 non-n